# RNN, LSTM, and Transformer

In this notebook, you will implement three fundamental sequence modeling architectures:
1. **Recurrent Neural Network (RNN)** - The foundation of sequence modeling
2. **Long Short-Term Memory (LSTM)** - Addressing the vanishing gradient problem
3. **Transformer** - The attention-based revolution

### Complete `# TODO`


### 📚 Prerequisites
- Basic understanding of neural networks
- Familiarity with PyTorch basics
- Knowledge of RNN, LSTM, and Transformer concepts

---

## Part 0: Setup and Data Preparation

In [ ]:
# Install and import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import math
import matplotlib.pyplot as plt
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

### Dataset: Sentiment Classification

We'll use a simple sentiment classification task. The goal is to classify movie reviews as positive or negative.

We'll create a synthetic dataset to avoid downloading large files and ensure the notebook runs quickly.

In [ ]:
# Sample sentiment data (simplified for educational purposes)
positive_reviews = [
    "this movie is great and wonderful",
    "excellent film with amazing acting",
    "i loved this movie so much",
    "brilliant performance by all actors",
    "fantastic story and great direction",
    "wonderful movie highly recommend it",
    "best film i have ever seen",
    "amazing cinematography and great plot",
    "superb acting and beautiful story",
    "loved every moment of this film",
    "incredible movie with perfect ending",
    "outstanding performance truly amazing work",
    "great movie will watch again",
    "excellent story telling and direction",
    "beautiful film with great message",
    "perfect movie for the weekend",
    "loved the characters and story",
    "amazing film must watch it",
    "great acting and wonderful plot",
    "superb movie highly entertaining film"
]

negative_reviews = [
    "this movie is terrible and boring",
    "worst film i have ever seen",
    "terrible acting and bad story",
    "boring movie waste of time",
    "awful film do not watch",
    "horrible movie very disappointing experience",
    "bad acting and terrible direction",
    "worst movie of the year",
    "disappointing film not worth watching",
    "terrible story and boring plot",
    "awful experience waste of money",
    "bad movie with poor acting",
    "horrible film very bad ending",
    "worst performance by the actors",
    "boring and terrible waste time",
    "disappointing movie bad story line",
    "awful acting and horrible plot",
    "terrible movie do not recommend",
    "bad film very disappointing watch",
    "worst movie horrible experience overall"
]

# Combine data
texts = positive_reviews + negative_reviews
labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)  # 1: positive, 0: negative

print(f"Total samples: {len(texts)}")
print(f"Positive: {sum(labels)}, Negative: {len(labels) - sum(labels)}")

In [ ]:
# Build vocabulary
class Vocabulary:
    def __init__(self):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.word_count = Counter()

    def build_vocab(self, texts):
        for text in texts:
            for word in text.lower().split():
                self.word_count[word] += 1

        for word in self.word_count:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def encode(self, text, max_len):
        tokens = text.lower().split()
        encoded = [self.word2idx.get(word, 1) for word in tokens]  # 1 is <UNK>
        # Pad or truncate
        if len(encoded) < max_len:
            encoded += [0] * (max_len - len(encoded))  # 0 is <PAD>
        else:
            encoded = encoded[:max_len]
        return encoded

    def __len__(self):
        return len(self.word2idx)

# Build vocabulary
vocab = Vocabulary()
vocab.build_vocab(texts)
print(f"Vocabulary size: {len(vocab)}")
print(f"Sample encoding: '{texts[0]}' -> {vocab.encode(texts[0], 10)}")

In [ ]:
# Create Dataset and DataLoader
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=10):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoded = self.vocab.encode(text, self.max_len)
        return torch.tensor(encoded), torch.tensor(label, dtype=torch.float32)

# Configuration
MAX_LEN = 10
BATCH_SIZE = 8
EMBED_DIM = 32
HIDDEN_DIM = 64

# Create dataset and dataloader
dataset = SentimentDataset(texts, labels, vocab, MAX_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Test dataloader
sample_batch = next(iter(dataloader))
print(f"Batch input shape: {sample_batch[0].shape}")  # [batch_size, seq_len]
print(f"Batch label shape: {sample_batch[1].shape}")  # [batch_size]

---

## Part 1: Recurrent Neural Network (RNN)

### Theory Recap

A simple RNN processes sequences by maintaining a hidden state that is updated at each time step:

$$h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh})$$

Where:
- $x_t$ is the input at time step $t$
- $h_t$ is the hidden state at time step $t$
- $W_{ih}$, $W_{hh}$ are weight matrices
- $b_{ih}$, $b_{hh}$ are bias vectors



### Task 1.1: Implement an RNN Cell from Scratch

In [ ]:
class RNNCellFromScratch(nn.Module):
    """
    A simple RNN cell implemented from scratch.

    Args:
        input_size: Size of input features
        hidden_size: Size of hidden state
    """
    def __init__(self, input_size, hidden_size):
        super(RNNCellFromScratch, self).__init__()
        self.hidden_size = hidden_size

        # TODO: Define the weight matrices and biases
        # Hint: You need W_ih (input to hidden), W_hh (hidden to hidden)
        # and their corresponding biases b_ih, b_hh
        # Use nn.Parameter to make them learnable

        # self.W_ih = nn.Parameter(...)
        # self.W_hh = nn.Parameter(...)
        # self.b_ih = nn.Parameter(...)
        # self.b_hh = nn.Parameter(...)

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the weight matrices and biases")

        # =========================================

    def forward(self, x, h_prev):
        """
        Forward pass for one time step.

        Args:
            x: Input tensor of shape (batch_size, input_size)
            h_prev: Previous hidden state of shape (batch_size, hidden_size)

        Returns:
            h_new: New hidden state of shape (batch_size, hidden_size)
        """
        # TODO: Implement the RNN cell forward pass
        # Formula: h_new = tanh(x @ W_ih.T + b_ih + h_prev @ W_hh.T + b_hh)

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the RNN forward pass")

        # =========================================

        return h_new

    def init_hidden(self, batch_size):
        """Initialize hidden state with zeros."""
        return torch.zeros(batch_size, self.hidden_size)

In [ ]:
# Test your RNN cell implementation
def test_rnn_cell():
    rnn_cell = RNNCellFromScratch(input_size=32, hidden_size=64)

    # Test input
    batch_size = 4
    x = torch.randn(batch_size, 32)
    h_prev = rnn_cell.init_hidden(batch_size)

    # Forward pass
    h_new = rnn_cell(x, h_prev)

    assert h_new.shape == (batch_size, 64), f"Expected shape (4, 64), got {h_new.shape}"
    assert torch.all(h_new >= -1) and torch.all(h_new <= 1), "Output should be in [-1, 1] (tanh)"
    print("✅ RNN Cell test passed!")
    print(f"   Output shape: {h_new.shape}")
    print(f"   Output range: [{h_new.min():.4f}, {h_new.max():.4f}]")

# Uncomment to test after implementing
# test_rnn_cell()

### Task 1.2: Build a Complete RNN Model for Sentiment Classification

In [ ]:
class RNNClassifier(nn.Module):
    """
    RNN-based text classifier for sentiment analysis.

    Architecture:
    1. Embedding layer: Convert word indices to dense vectors
    2. RNN layer: Process sequence and get final hidden state
    3. Fully connected layer: Map hidden state to output
    4. Sigmoid: Convert to probability
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim=1, pad_idx=0):
        super(RNNClassifier, self).__init__()

        # TODO: Define the layers
        # 1. Embedding layer with padding_idx
        # 2. RNN layer (you can use nn.RNN here for simplicity)
        # 3. Fully connected layer

        # Hint:
        # self.embedding = nn.Embedding(...)
        # self.rnn = nn.RNN(..., batch_first=True)
        # self.fc = nn.Linear(...)

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the RNN classifier layers")

        # =========================================

    def forward(self, x):
        """
        Forward pass.

        Args:
            x: Input tensor of shape (batch_size, seq_len)

        Returns:
            output: Probability tensor of shape (batch_size,)
        """
        # TODO: Implement the forward pass
        # 1. Pass through embedding layer
        # 2. Pass through RNN (get the final hidden state)
        # 3. Pass through fully connected layer
        # 4. Apply sigmoid activation

        # Hint: nn.RNN returns (output, hidden)
        # - output shape: (batch_size, seq_len, hidden_dim)
        # - hidden shape: (num_layers, batch_size, hidden_dim)
        # Use hidden[-1] to get the last layer's hidden state

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the RNN classifier forward pass")

        # =========================================

        return output

In [ ]:
# Test your RNN classifier
def test_rnn_classifier():
    model = RNNClassifier(
        vocab_size=len(vocab),
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM
    )

    # Test with a batch
    sample_input, sample_label = next(iter(dataloader))
    output = model(sample_input)

    assert output.shape == (BATCH_SIZE,), f"Expected shape ({BATCH_SIZE},), got {output.shape}"
    assert torch.all(output >= 0) and torch.all(output <= 1), "Output should be in [0, 1] (sigmoid)"
    print("✅ RNN Classifier test passed!")
    print(f"   Input shape: {sample_input.shape}")
    print(f"   Output shape: {output.shape}")
    print(f"   Sample predictions: {output[:3].detach().numpy()}")

# Uncomment to test after implementing
# test_rnn_classifier()

---

## Part 2: Long Short-Term Memory (LSTM)

### Theory Recap

LSTM addresses the vanishing gradient problem by introducing gates:

- **Forget gate**: $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$
- **Input gate**: $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$
- **Cell candidate**: $\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$
- **Cell state**: $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$
- **Output gate**: $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$
- **Hidden state**: $h_t = o_t \odot \tanh(C_t)$

### Task 2.1: Implement an LSTM Cell from Scratch

In [ ]:
class LSTMCellFromScratch(nn.Module):
    """
    An LSTM cell implemented from scratch.

    Args:
        input_size: Size of input features
        hidden_size: Size of hidden state
    """
    def __init__(self, input_size, hidden_size):
        super(LSTMCellFromScratch, self).__init__()
        self.hidden_size = hidden_size

        # TODO: Define the weight matrices for all four gates
        # Hint: A common approach is to compute all gates together:
        # - W_i: weight for input (combines both x and h)
        # - b_i: bias
        # Size of W_i should be (4 * hidden_size, input_size + hidden_size)
        # This computes [i, f, g, o] all at once

        # Alternative: Define separate weights for each gate
        # W_ii, W_hi (input gate)
        # W_if, W_hf (forget gate)
        # W_ig, W_hg (cell gate)
        # W_io, W_ho (output gate)

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the LSTM weight matrices")

        # =========================================

    def forward(self, x, hidden):
        """
        Forward pass for one time step.

        Args:
            x: Input tensor of shape (batch_size, input_size)
            hidden: Tuple of (h_prev, c_prev)
                h_prev: Previous hidden state (batch_size, hidden_size)
                c_prev: Previous cell state (batch_size, hidden_size)

        Returns:
            h_new: New hidden state (batch_size, hidden_size)
            c_new: New cell state (batch_size, hidden_size)
        """
        h_prev, c_prev = hidden

        # TODO: Implement the LSTM forward pass
        # 1. Compute the input gate: i = sigmoid(W_ii @ x + W_hi @ h_prev + b_i)
        # 2. Compute the forget gate: f = sigmoid(W_if @ x + W_hf @ h_prev + b_f)
        # 3. Compute the cell candidate: g = tanh(W_ig @ x + W_hg @ h_prev + b_g)
        # 4. Compute the output gate: o = sigmoid(W_io @ x + W_ho @ h_prev + b_o)
        # 5. Compute new cell state: c_new = f * c_prev + i * g
        # 6. Compute new hidden state: h_new = o * tanh(c_new)

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the LSTM forward pass")

        # =========================================

        return h_new, c_new

    def init_hidden(self, batch_size):
        """Initialize hidden state and cell state with zeros."""
        h0 = torch.zeros(batch_size, self.hidden_size)
        c0 = torch.zeros(batch_size, self.hidden_size)
        return (h0, c0)

In [ ]:
# Test your LSTM cell implementation
def test_lstm_cell():
    lstm_cell = LSTMCellFromScratch(input_size=32, hidden_size=64)

    # Test input
    batch_size = 4
    x = torch.randn(batch_size, 32)
    hidden = lstm_cell.init_hidden(batch_size)

    # Forward pass
    h_new, c_new = lstm_cell(x, hidden)

    assert h_new.shape == (batch_size, 64), f"h_new: Expected shape (4, 64), got {h_new.shape}"
    assert c_new.shape == (batch_size, 64), f"c_new: Expected shape (4, 64), got {c_new.shape}"
    print("✅ LSTM Cell test passed!")
    print(f"   h_new shape: {h_new.shape}, range: [{h_new.min():.4f}, {h_new.max():.4f}]")
    print(f"   c_new shape: {c_new.shape}, range: [{c_new.min():.4f}, {c_new.max():.4f}]")

# Uncomment to test after implementing
# test_lstm_cell()

### Task 2.2: Build a Complete LSTM Model for Sentiment Classification

In [ ]:
class LSTMClassifier(nn.Module):
    """
    LSTM-based text classifier for sentiment analysis.

    Architecture:
    1. Embedding layer: Convert word indices to dense vectors
    2. LSTM layer: Process sequence with memory
    3. Fully connected layer: Map hidden state to output
    4. Sigmoid: Convert to probability
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim=1,
                 num_layers=1, dropout=0.0, pad_idx=0):
        super(LSTMClassifier, self).__init__()

        # TODO: Define the layers
        # 1. Embedding layer with padding_idx
        # 2. LSTM layer with specified num_layers and dropout
        # 3. Dropout layer (for regularization)
        # 4. Fully connected layer

        # Hint:
        # self.embedding = nn.Embedding(...)
        # self.lstm = nn.LSTM(..., batch_first=True, num_layers=..., dropout=...)
        # self.dropout = nn.Dropout(...)
        # self.fc = nn.Linear(...)

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the LSTM classifier layers")

        # =========================================

    def forward(self, x):
        """
        Forward pass.

        Args:
            x: Input tensor of shape (batch_size, seq_len)

        Returns:
            output: Probability tensor of shape (batch_size,)
        """
        # TODO: Implement the forward pass
        # 1. Embedding: (batch_size, seq_len) -> (batch_size, seq_len, embed_dim)
        # 2. LSTM: -> output, (h_n, c_n)
        # 3. Take final hidden state h_n[-1]: (batch_size, hidden_dim)
        # 4. Apply dropout
        # 5. Fully connected layer
        # 6. Sigmoid activation

        # Hint: nn.LSTM returns (output, (h_n, c_n))
        # h_n shape: (num_layers, batch_size, hidden_dim)

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the LSTM classifier forward pass")

        # =========================================

        return output

In [ ]:
# Test your LSTM classifier
def test_lstm_classifier():
    model = LSTMClassifier(
        vocab_size=len(vocab),
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=2,
        dropout=0.3
    )

    # Test with a batch
    sample_input, sample_label = next(iter(dataloader))
    model.eval()  # Set to eval mode to disable dropout for testing
    with torch.no_grad():
        output = model(sample_input)

    assert output.shape == (BATCH_SIZE,), f"Expected shape ({BATCH_SIZE},), got {output.shape}"
    assert torch.all(output >= 0) and torch.all(output <= 1), "Output should be in [0, 1] (sigmoid)"
    print("✅ LSTM Classifier test passed!")
    print(f"   Input shape: {sample_input.shape}")
    print(f"   Output shape: {output.shape}")
    print(f"   Sample predictions: {output[:3].numpy()}")

# Uncomment to test after implementing
# test_lstm_classifier()

---

## Part 3: Transformer (Self-Attention)

### Theory Recap

The Transformer architecture relies on the **Self-Attention** mechanism:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $Q$ (Query), $K$ (Key), $V$ (Value) are linear projections of the input
- $d_k$ is the dimension of the keys (for scaling)

**Multi-Head Attention** allows the model to jointly attend to information from different representation subspaces.



### Task 3.1: Implement Scaled Dot-Product Attention

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Compute scaled dot-product attention.

    Args:
        query: Tensor of shape (batch_size, num_heads, seq_len, d_k)
        key: Tensor of shape (batch_size, num_heads, seq_len, d_k)
        value: Tensor of shape (batch_size, num_heads, seq_len, d_v)
        mask: Optional mask tensor of shape (batch_size, 1, 1, seq_len) or (batch_size, 1, seq_len, seq_len)

    Returns:
        output: Tensor of shape (batch_size, num_heads, seq_len, d_v)
        attention_weights: Tensor of shape (batch_size, num_heads, seq_len, seq_len)
    """
    d_k = query.size(-1)

    # TODO: Implement scaled dot-product attention
    # Step 1: Compute attention scores = Q @ K^T / sqrt(d_k)
    # Step 2: Apply mask if provided (set masked positions to -inf before softmax)
    # Step 3: Apply softmax to get attention weights
    # Step 4: Multiply attention weights with values

    # Hint for matrix multiplication with batched tensors:
    # Use torch.matmul() or @ operator
    # key.transpose(-2, -1) to transpose the last two dimensions

    # ============ YOUR CODE HERE ============

    raise NotImplementedError("Implement scaled dot-product attention")

    # =========================================

    return output, attention_weights

In [ ]:
# Test scaled dot-product attention
def test_attention():
    batch_size, num_heads, seq_len, d_k = 2, 4, 10, 16

    query = torch.randn(batch_size, num_heads, seq_len, d_k)
    key = torch.randn(batch_size, num_heads, seq_len, d_k)
    value = torch.randn(batch_size, num_heads, seq_len, d_k)

    output, attn_weights = scaled_dot_product_attention(query, key, value)

    assert output.shape == (batch_size, num_heads, seq_len, d_k), f"Output shape mismatch"
    assert attn_weights.shape == (batch_size, num_heads, seq_len, seq_len), f"Attention weights shape mismatch"

    # Check attention weights sum to 1
    weight_sums = attn_weights.sum(dim=-1)
    assert torch.allclose(weight_sums, torch.ones_like(weight_sums), atol=1e-5), "Attention weights should sum to 1"

    print("✅ Scaled dot-product attention test passed!")
    print(f"   Output shape: {output.shape}")
    print(f"   Attention weights shape: {attn_weights.shape}")

# Uncomment to test after implementing
# test_attention()

### Task 3.2: Implement Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention module.

    Args:
        d_model: Model dimension
        num_heads: Number of attention heads
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # TODO: Define the linear projection layers
        # We need:
        # - W_q: Projects input to queries
        # - W_k: Projects input to keys
        # - W_v: Projects input to values
        # - W_o: Projects concatenated heads back to d_model

        # Each of W_q, W_k, W_v has shape (d_model, d_model)
        # W_o also has shape (d_model, d_model)

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the linear projection layers")

        # =========================================

    def forward(self, query, key, value, mask=None):
        """
        Forward pass.

        Args:
            query: (batch_size, seq_len, d_model)
            key: (batch_size, seq_len, d_model)
            value: (batch_size, seq_len, d_model)
            mask: Optional mask

        Returns:
            output: (batch_size, seq_len, d_model)
        """
        batch_size = query.size(0)

        # TODO: Implement multi-head attention forward pass
        # Step 1: Linear projections to get Q, K, V
        # Step 2: Reshape to (batch_size, num_heads, seq_len, d_k)
        #         Hint: Use view and transpose
        #         x.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
        # Step 3: Apply scaled dot-product attention (use your function from Task 3.1)
        # Step 4: Concatenate heads: transpose back and reshape to (batch_size, seq_len, d_model)
        # Step 5: Final linear projection with W_o

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the multi-head attention forward pass")

        # =========================================

        return output

In [ ]:
# Test multi-head attention
def test_multi_head_attention():
    d_model, num_heads = 64, 8
    batch_size, seq_len = 4, 10

    mha = MultiHeadAttention(d_model, num_heads)

    x = torch.randn(batch_size, seq_len, d_model)
    output = mha(x, x, x)  # Self-attention

    assert output.shape == (batch_size, seq_len, d_model), f"Expected shape {(batch_size, seq_len, d_model)}, got {output.shape}"
    print("✅ Multi-Head Attention test passed!")
    print(f"   Input shape: {x.shape}")
    print(f"   Output shape: {output.shape}")

# Uncomment to test after implementing
# test_multi_head_attention()

### Task 3.3: Implement a Transformer Encoder Block

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Positional Encoding using sine and cosine functions.
    This is provided for you - no need to implement.
    """
    def __init__(self, d_model, max_len=512):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)

        self.register_buffer('pe', pe)

    def forward(self, x):
        """Add positional encoding to input."""
        return x + self.pe[:, :x.size(1), :]

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """
    A single Transformer Encoder block.

    Architecture:
    1. Multi-Head Self-Attention
    2. Add & Norm (residual connection + layer normalization)
    3. Feed-Forward Network
    4. Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerEncoderBlock, self).__init__()

        # TODO: Define the layers
        # 1. Multi-head attention (use your MultiHeadAttention class)
        # 2. Layer normalization (2 instances needed)
        # 3. Feed-forward network: Linear -> ReLU -> Linear
        # 4. Dropout (for regularization)

        # Hint:
        # self.attention = MultiHeadAttention(...)
        # self.norm1 = nn.LayerNorm(...)
        # self.norm2 = nn.LayerNorm(...)
        # self.feed_forward = nn.Sequential(
        #     nn.Linear(d_model, d_ff),
        #     nn.ReLU(),
        #     nn.Linear(d_ff, d_model)
        # )
        # self.dropout = nn.Dropout(...)

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the Transformer encoder block layers")

        # =========================================

    def forward(self, x, mask=None):
        """
        Forward pass.

        Args:
            x: Input tensor (batch_size, seq_len, d_model)
            mask: Optional attention mask

        Returns:
            output: (batch_size, seq_len, d_model)
        """
        # TODO: Implement the forward pass
        # 1. Self-attention: attn_output = attention(x, x, x, mask)
        # 2. Add & Norm: x = norm1(x + dropout(attn_output))
        # 3. Feed-forward: ff_output = feed_forward(x)
        # 4. Add & Norm: x = norm2(x + dropout(ff_output))

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the Transformer encoder block forward pass")

        # =========================================

        return x

In [ ]:
# Test transformer encoder block
def test_transformer_block():
    d_model, num_heads, d_ff = 64, 8, 256
    batch_size, seq_len = 4, 10

    encoder_block = TransformerEncoderBlock(d_model, num_heads, d_ff)

    x = torch.randn(batch_size, seq_len, d_model)
    output = encoder_block(x)

    assert output.shape == (batch_size, seq_len, d_model), f"Shape mismatch"
    print("✅ Transformer Encoder Block test passed!")
    print(f"   Input shape: {x.shape}")
    print(f"   Output shape: {output.shape}")

# Uncomment to test after implementing
# test_transformer_block()

### Task 3.4: Build a Complete Transformer Model for Sentiment Classification

In [ ]:
class TransformerClassifier(nn.Module):
    """
    Transformer-based text classifier for sentiment analysis.

    Architecture:
    1. Embedding layer + Positional Encoding
    2. Stack of Transformer Encoder blocks
    3. Global average pooling (or use [CLS] token)
    4. Fully connected layer + Sigmoid
    """
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 max_len=512, dropout=0.1, pad_idx=0):
        super(TransformerClassifier, self).__init__()

        self.d_model = d_model

        # TODO: Define the layers
        # 1. Embedding layer with padding_idx
        # 2. Positional encoding (use the provided PositionalEncoding class)
        # 3. Dropout layer
        # 4. Stack of Transformer encoder blocks (use nn.ModuleList)
        # 5. Final normalization layer
        # 6. Fully connected output layer

        # Hint:
        # self.embedding = nn.Embedding(...)
        # self.pos_encoding = PositionalEncoding(...)
        # self.dropout = nn.Dropout(...)
        # self.encoder_blocks = nn.ModuleList([TransformerEncoderBlock(...) for _ in range(num_layers)])
        # self.norm = nn.LayerNorm(...)
        # self.fc = nn.Linear(...)

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the Transformer classifier layers")

        # =========================================

    def forward(self, x, mask=None):
        """
        Forward pass.

        Args:
            x: Input tensor (batch_size, seq_len)
            mask: Optional attention mask

        Returns:
            output: Probability tensor (batch_size,)
        """
        # TODO: Implement the forward pass
        # 1. Embedding: scale by sqrt(d_model) as in original paper
        # 2. Add positional encoding
        # 3. Apply dropout
        # 4. Pass through each encoder block
        # 5. Apply final normalization
        # 6. Global average pooling over sequence dimension
        # 7. Fully connected layer
        # 8. Sigmoid activation

        # Hint for global average pooling:
        # x.mean(dim=1) to average over the sequence dimension

        # ============ YOUR CODE HERE ============

        raise NotImplementedError("Implement the Transformer classifier forward pass")

        # =========================================

        return output

In [ ]:
# Test your Transformer classifier
def test_transformer_classifier():
    model = TransformerClassifier(
        vocab_size=len(vocab),
        d_model=64,
        num_heads=4,
        d_ff=128,
        num_layers=2,
        max_len=MAX_LEN,
        dropout=0.1
    )

    # Test with a batch
    sample_input, sample_label = next(iter(dataloader))
    model.eval()
    with torch.no_grad():
        output = model(sample_input)

    assert output.shape == (BATCH_SIZE,), f"Expected shape ({BATCH_SIZE},), got {output.shape}"
    assert torch.all(output >= 0) and torch.all(output <= 1), "Output should be in [0, 1]"
    print("✅ Transformer Classifier test passed!")
    print(f"   Input shape: {sample_input.shape}")
    print(f"   Output shape: {output.shape}")
    print(f"   Sample predictions: {output[:3].numpy()}")

# Uncomment to test after implementing
# test_transformer_classifier()

---

## Part 4: Training and Plots



In [ ]:
def train_model(model, dataloader, epochs=50, lr=0.001):
    """
    Train a model and return training history.
    """
    model = model.to(device)
    criterion = nn.MSELoss()  #nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'loss': [], 'accuracy': []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            predictions = (outputs > 0.5).float()
            correct += (predictions == batch_y).sum().item()
            total += batch_y.size(0)

        avg_loss = total_loss / len(dataloader)
        accuracy = correct / total
        history['loss'].append(avg_loss)
        history['accuracy'].append(accuracy)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Accuracy: {accuracy:.4f}")

    return history

In [ ]:
#Train all models

print("=" * 50)
print("Training RNN Classifier")
print("=" * 50)
rnn_model = RNNClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM)
rnn_history = train_model(rnn_model, dataloader, epochs=100)

print("\n" + "=" * 50)
print("Training LSTM Classifier")
print("=" * 50)
lstm_model = LSTMClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM, num_layers=1)
lstm_history = train_model(lstm_model, dataloader, epochs=100)

print("\n" + "=" * 50)
print("Training Transformer Classifier")
print("=" * 50)
transformer_model = TransformerClassifier(
    vocab_size=len(vocab),
    d_model=64,
    num_heads=4,
    d_ff=128,
    num_layers=2
)
transformer_history = train_model(transformer_model, dataloader, epochs=100)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(rnn_history['loss'], label='RNN', marker='o', markevery=5)
axes[0].plot(lstm_history['loss'], label='LSTM', marker='s', markevery=5)
axes[0].plot(transformer_history['loss'], label='Transformer', marker='^', markevery=5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(rnn_history['accuracy'], label='RNN', marker='o', markevery=5)
axes[1].plot(lstm_history['accuracy'], label='LSTM', marker='s', markevery=5)
axes[1].plot(transformer_history['accuracy'], label='Transformer', marker='^', markevery=5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()